# Student Performance Prediction System

## Project Overview

This project develops a machine learning system to predict a student's final mathematics grade (`G3`) using demographic, social, academic, and behavioral features from the UCI Student Performance dataset.

### Workflow

**Data → Cleaning → Analysis → Feature Selection → Preprocessing → Training → Testing → Evaluation → Prediction**

The project compares multiple regression models and evaluates their performance using MAE, MSE, RMSE, and R².


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "student-mat.csv"

print("Project directory:", BASE_DIR)
print("Dataset path:", DATA_PATH)


## 2. Load the Dataset

The project uses the mathematics subset of the UCI Student Performance dataset.

The dataset contains **395 student records and 33 columns**.

The target variable is `G3`, representing the final mathematics grade.


In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")

print("Dataset shape:", df.shape)
display(df.head())


## 3. Dataset Understanding

We inspect:

- Dataset dimensions
- Column names
- Data types
- Missing values
- Duplicate records
- Statistical summary


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe())


## 4. Data Cleaning

The dataset was checked for missing values and duplicate records.

The original dataset contains **no missing values and no duplicate rows**, so no rows needed to be removed during cleaning.


In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


## 5. Exploratory Data Analysis

Important relationships identified during analysis include:

- `G2` has a strong positive relationship with `G3`.
- `G1` also has a strong positive relationship with `G3`.
- Students with more previous failures generally have lower final grades.
- Study time shows a positive but comparatively weaker relationship with final grade.
- Absences have a relatively weak linear relationship with `G3`.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["G3"], bins=11)
plt.title("Distribution of Final Grade (G3)")
plt.xlabel("G3")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()


In [ ]:
correlations = df.select_dtypes(include=np.number).corr()["G3"].sort_values(ascending=False)

print("Numerical correlations with G3:")
display(correlations)


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["G2"], df["G3"], alpha=0.7)
plt.title("G2 vs G3")
plt.xlabel("G2")
plt.ylabel("G3")
plt.tight_layout()
plt.show()


## 6. Feature Selection

The target variable is:

`G3`

All other columns are used as input features.

The dataset contains both numerical and categorical features. Categorical variables are encoded using One-Hot Encoding while numerical variables are passed through without scaling.


In [ ]:
X = df.drop(columns=["G3"])
y = df["G3"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))


## 7. Preprocessing and Train/Test Split

The data is divided into:

- **80% training data**
- **20% testing data**

A `ColumnTransformer` applies One-Hot Encoding to categorical variables while retaining numerical variables.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 8. Model Training

The project evaluates four regression models:

1. Linear Regression
2. Decision Tree Regression
3. Random Forest Regression
4. Tuned Decision Tree and Random Forest models

The tree-based models were also evaluated using cross-validation and hyperparameter tuning.


In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=5,
        min_samples_split=5,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1
    )
}

model_results = []

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mse = mean_squared_error(y_test, predictions)

    model_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(model_results)
display(results_df)


## 9. Final Model Comparison

The final comparison includes the fixed tree-based models and the tuned models evaluated during the project.

The previously generated final results are loaded below.


In [ ]:
RESULTS_PATH = BASE_DIR / "reports" / "final_model_results.csv"

final_results = pd.read_csv(RESULTS_PATH)

display(final_results)


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(final_results["Model"], final_results["RMSE"])
plt.title("Final Model Comparison - RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.xticks(rotation=25)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(final_results["Model"], final_results["R2"])
plt.title("Final Model Comparison - R2 Score")
plt.xlabel("Model")
plt.ylabel("R2 Score")
plt.xticks(rotation=25)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Final Prediction

The project also includes a reusable prediction workflow in:

`src/predict_student.py`

The prediction workflow validates the required input features, generates predictions, checks that the predicted grade falls within the expected 0–20 range, and handles invalid input.


In [ ]:
# Example student profile
sample_student = X.iloc[[0]]

print("Sample student input:")
display(sample_student)

print("\nActual G3:", y.iloc[0])


## 11. Conclusion

The project demonstrates a complete machine learning workflow for student performance prediction, including data inspection, cleaning, exploratory analysis, feature preparation, model training, evaluation, hyperparameter tuning, and prediction validation.

Model performance is evaluated using:

- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- R² Score

### Important Modeling Note

`G1` and `G2` are previous-period grades and are included as input features. Therefore, the resulting system should be understood as predicting the final grade using earlier academic performance information, rather than predicting performance before any prior grades are available.
